<center>
    <p style="text-align:center">
    <img alt="arize logo" src="https://storage.googleapis.com/arize-assets/arize-logo-white.jpg" width="300"/>
        <br>
        <a href="https://docs.arize.com/arize/">Docs</a>
        |
        <a href="https://github.com/Arize-ai">GitHub</a>
    </p>
</center>

# Build, Test, and Optimize a Trip-Planner Prompt

An end-to-end walkthrough of the prompt iteration cycle in Arize AX, using a trip-planner use case.

You'll work through three sections that mirror the [Prompts concept docs](https://arize.com/docs/ax/concepts/prompts/overview):

1. **Create** — define a Prompt Object and save it to Prompt Hub.
1. **Test** — run the prompt against a dataset with an evaluator, as an Arize experiment.
1. **Optimize** — iterate the prompt, save a new version, and compare runs side by side.

By the end you'll have a versioned trip-planner prompt in Prompt Hub, two experiment runs against the same dataset, and a sense of where each version did better. Total cost: a few cents on `gpt-5.4-mini`.

## What you'll learn

This notebook is a hands-on companion to the prompt concepts docs. As you work through it, you'll see:

- **What a Prompt Object actually is in code** — not a string, but a bundle of template, model, invocation parameters, tools, and response format. We'll create one with the `arize` SDK and save it to Prompt Hub.
- **Why versioning matters in practice** — every save creates a new immutable version. Tags like `production` point at specific versions and can be moved without touching code.
- **How experiments make prompts comparable** — running the same dataset through two prompt versions, with the same evaluator, produces a side-by-side score delta you can act on.
- **Why the evaluator is half the loop** — the evaluator decides which prompt changes count as improvements. A loose evaluator scores everything well and hides regressions; a strict one discriminates between a vague prompt and a disciplined one. The deterministic evaluator we'll write here is strict on format, so the difference between v1 and v2 shows up clearly.

The goal isn't just to run the cells — it's to understand the iteration loop well enough that you can apply it to your own prompts.

## Setup

You'll need three credentials to run this notebook:

- **`ARIZE_API_KEY`** — your developer key for the Arize AX API, found at [app.arize.com/admin](https://app.arize.com/admin) under **API Keys**.
- **`ARIZE_SPACE_ID`** — the workspace your prompts, datasets, and experiments will land in. From the same admin page or the URL bar in the AX UI.
- **`OPENAI_API_KEY`** — used by the task we'll build to call `gpt-5.4-mini`. The Prompt Object holds the model identifier; the LLM call itself is made directly by your task code via OpenAI's SDK.

The cells below install dependencies and pick up keys from environment variables when present, falling back to interactive prompts otherwise.

In [ ]:
%pip install -qq arize openai pandas ipywidgets

In [ ]:
import os
from getpass import getpass

# Prefer env vars so the notebook can run non-interactively (e.g., in CI). Fall back to
# getpass so first-time readers can paste credentials without editing the file.
ARIZE_API_KEY = os.environ.get("ARIZE_API_KEY") or getpass("🔑 Enter your Arize API key: ")
ARIZE_SPACE_ID = os.environ.get("ARIZE_SPACE_ID") or getpass("🔑 Enter your Arize space ID: ")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass("🔑 Enter your OpenAI API key: ")

# The OpenAI SDK reads OPENAI_API_KEY from the environment; make sure it's set even when
# getpass collected the value above.
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from arize.client import ArizeClient
from uuid import uuid4

# The Arize client is the entry point to every AX resource — prompts, datasets, experiments,
# evaluators, etc. all hang off it as namespaces (e.g., client.prompts.create).
client = ArizeClient(api_key=ARIZE_API_KEY)

# We tag every artifact this notebook creates with a unique RUN_ID. That way you can rerun
# the notebook end-to-end without colliding with prompts or datasets from a previous run,
# and your Arize workspace gets a clean trail of what each pass produced.
RUN_ID = uuid4().hex[:8]
print(f"Run ID: {RUN_ID}")

## Section 1 — Create the prompt

A trip planner takes a destination, duration, travel style, and some research/budget/local context, and produces a structured day-by-day itinerary.

Our **Prompt Object** has five parts:

- **Prompt template** — system + user messages with `{variable}` placeholders.
- **Model** — `gpt-5.4-mini`.
- **Invocation parameters** — temperature, max tokens, etc.
- **Tools** — none for this prompt.
- **Response format** — plain text in this version (we'll tighten this later).

See [The Prompt Object](https://arize.com/docs/ax/concepts/prompts/prompt-object) for the conceptual breakdown.

### Why we start with a deliberately weak v1

A casual first prompt is the realistic starting point for most teams: a one-line role description plus the user's question. We're going to make exactly that, ship it, score it with an evaluator, then improve it in Section 3. The point is to *show* the iteration loop — not to write a perfect prompt on the first try.

Notice what's **not** in v1's system message: no format constraints, no example output, no instructions about times or costs. The model is left to figure out the shape. That matters in Section 3.

In [ ]:
from arize.prompts.types import InputVariableFormat, InvocationParams, LlmProvider, LLMMessage, MessageRole

PROMPT_NAME = f"trip-planner-{RUN_ID}"

# v1 system message: deliberately vague — no format constraints, no examples, no rules
# about what to include. This is the "first attempt" most teams ship.
system_message_v1 = (
    "You are a helpful travel planner. Given a destination, duration, and travel style, "
    "produce a trip plan the user can follow."
)

# The user template is the part of the prompt that gets filled in per request. The
# {placeholders} match column names in the dataset we'll build in Section 2 — at run time
# the experiment substitutes the row's value for each placeholder.
user_message = (
    "Plan a {duration} trip to {destination}. Travel style: {travel_style}.\n\n"
    "Research: {research}\n"
    "Budget: {budget_info}\n"
    "Local notes: {local_info}"
)

# client.prompts.create writes a new Prompt Object to Prompt Hub. Every save produces an
# immutable, hashed version that lives in history forever — you can always load this exact
# version later, even after edits.
prompt_v1 = client.prompts.create(
    space=ARIZE_SPACE_ID,
    name=PROMPT_NAME,
    description="Day-by-day itinerary generator for a given destination, duration, and travel style.",
    commit_message="v1: baseline trip planner",  # like a git commit message — shows up in the UI's version history
    input_variable_format=InputVariableFormat.F_STRING,  # {name} placeholders (Python f-string style); MUSTACHE is the {{name}} alternative
    provider=LlmProvider.OPEN_AI,
    model="gpt-5.4-mini",  # The model is part of the Prompt Object — changing it is a new version
    messages=[
        # System message sets the assistant's role; user message carries the per-request payload.
        LLMMessage(role=MessageRole.SYSTEM, content=system_message_v1),
        LLMMessage(role=MessageRole.USER, content=user_message),
    ],
    invocation_params=InvocationParams(max_completion_tokens=600),
)

# The returned PromptWithVersion exposes the prompt's metadata (name, id, description) at
# the top level, plus a nested .version with the immutable snapshot (messages, model,
# invocation_params, labels, etc.).
print(f"Created prompt {prompt_v1.name} (id={prompt_v1.id})")
print(f"Initial version: {prompt_v1.version.id}")

## Section 2 — Test the prompt against a dataset

Now we need data. We'll define a small dataset of trip-planner inputs covering different destinations and travel styles, upload it to Arize, and run the prompt against it as an experiment.

### What's an experiment, really?

An **experiment** in Arize AX is the unit of comparable measurement. It combines three things:

- A **dataset** of input rows — the same rows for every experiment, so scores can be compared apples-to-apples.
- A **task** — a Python function that consumes one row and returns one output. For prompt experiments, the task renders the prompt template and calls the LLM.
- One or more **evaluators** — functions that score the task's output. They return a label, a number, and an explanation per row.

Run the same dataset through two prompt versions with the same evaluator and you get a row-by-row score delta. That delta is the operational answer to *"is this prompt better?"*

See [Experiments for prompts](https://arize.com/docs/ax/concepts/prompts/experiments-for-prompts) for the full conceptual treatment.

In [ ]:
import pandas as pd

DATASET_NAME = f"trip-planner-test-set-{RUN_ID}"

# Each dict in this list is one dataset row. The keys must match the {placeholders} in the
# prompt's user_message — the experiment substitutes row[col] for {col} when it renders
# the prompt per row.
#
# Five rows is enough to demonstrate the comparison while keeping the API spend negligible.
# In production you'd build a much larger and more carefully-balanced golden dataset; for
# this tutorial we keep it small and varied (different durations, regions, travel styles)
# so the prompts have to handle range.
examples = [
    {
        "destination": "Istanbul, Turkey",
        "duration": "3 days",
        "travel_style": "standard",
        "research": "Best time to visit is April-May. Top attractions: Hagia Sophia, Blue Mosque, Grand Bazaar.",
        "budget_info": "Mid-range hotels $80-120/night, meals $15-30 each, transport $5-10/day.",
        "local_info": "Try Karaköy for breakfast spots; avoid Sultanahmet for dinner due to tourist markup.",
    },
    {
        "destination": "Bangkok, Thailand",
        "duration": "4 days",
        "travel_style": "family-friendly",
        "research": "Cool season Nov-Feb best. Top kid-friendly: Safari World, Dream World, river cruise.",
        "budget_info": "Family rooms $100-180/night, street food $2-5, taxis $5-15.",
        "local_info": "Skytrain reliable; visit temples early morning to beat heat.",
    },
    {
        "destination": "Barcelona, Spain",
        "duration": "5 days",
        "travel_style": "romantic",
        "research": "Spring/fall ideal. Sagrada Familia, Park Güell, tapas tours, beach access from city center.",
        "budget_info": "Boutique hotels $150-250/night, tapas dinner $30-50, metro $12 day pass.",
        "local_info": "Dinner starts late (9pm+); reserve Sagrada Familia tickets weeks ahead.",
    },
    {
        "destination": "Reykjavik, Iceland",
        "duration": "4 days",
        "travel_style": "adventure",
        "research": "Sept-Mar for northern lights; June-Aug for midnight sun. Blue Lagoon, Golden Circle, glacier tours.",
        "budget_info": "Hotels $200-300/night, dinner $40-70, day tours $80-150.",
        "local_info": "Rent a car for flexibility; weather changes fast — pack layers.",
    },
    {
        "destination": "New York City, USA",
        "duration": "2 days",
        "travel_style": "standard",
        "research": "Times Square, Central Park, museums (Met, MoMA), Brooklyn Bridge walk.",
        "budget_info": "Hotels $250-400/night, meals $20-50, subway $33 7-day pass.",
        "local_info": "Walk where possible; subway faster than taxis in midtown.",
    },
]

# client.datasets.create accepts a list of dicts or a pandas DataFrame. Each column in the
# DataFrame becomes a dataset column in Arize. The dataset is versioned — appending
# examples later creates a new version of the same dataset.
examples_df = pd.DataFrame(examples)
dataset = client.datasets.create(
    name=DATASET_NAME,
    space=ARIZE_SPACE_ID,
    examples=examples_df,
)
print(f"Created dataset {dataset.name} with {len(examples)} rows (id={dataset.id})")

**Open Arize AX in another tab** and navigate to the dataset that was just created. You'll see:

- The five rows in a table view, with each dict key as a column.
- An **Examples** tab with the raw data and an **Experiments** tab that will fill in as we run experiments below.
- A dataset ID and version number — datasets are versioned just like prompts, so this exact snapshot is recoverable later even if you append more examples.

Keep that tab open. By the end of this notebook, the Experiments tab will hold two runs (v1 and v2) you can compare side by side in the UI.

### Define the task

The **task** is the per-row workhorse of the experiment. The experiment runner walks every row of the dataset, hands the row to your task, and stores whatever the task returns as the output for that row.

For a prompt experiment, the contract is simple:

```
task(dataset_row: dict) -> str
```

Inside the task we:

1. **Render the user template** with the row's variables (`.format(**row)`).
2. **Call the LLM** with the system message and the rendered user message.
3. **Return the text output** — the experiment runner records it.

We're going to run this loop twice (once for v1, once for v2). To avoid duplicating the logic, we build a small factory `make_task` that takes the system message + user template and returns a ready-to-use task closure.

> **Production note:** in a real application you'd fetch the prompt from Prompt Hub at runtime — `client.prompts.get(prompt=PROMPT_NAME, label="production")` — and use the messages/model/parameters it returns. For a self-contained tutorial, we render the messages inline.

In [ ]:
from openai import OpenAI

# A single OpenAI client is reused across all task invocations.
oai = OpenAI()


def make_task(system_message: str, user_template: str, model: str = "gpt-5.4-mini"):
    """Build an experiment task that runs the trip-planner prompt against each row.

    Using a factory lets v1 and v2 share rendering + LLM-call logic. The only thing that
    differs between the two runs is the system message and (optionally) the model — both
    are captured by closure.
    """

    def task(dataset_row) -> str:
        # The experiment runner passes one dataset row per call. The row is dict-like,
        # so .format(**row) substitutes each {placeholder} with the row's column value.
        rendered_user = user_template.format(**dataset_row)

        # Standard chat-completions call. The Prompt Object's model and invocation params
        # would normally come from Prompt Hub; we inline them here to keep the cell
        # self-contained.
        resp = oai.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": rendered_user},
            ],
        )
        return resp.choices[0].message.content

    return task


# Bind v1's system message into a ready-to-run task. We'll do the same for v2 in Section 3.
task_v1 = make_task(system_message_v1, user_message)

### Define the evaluator

The **evaluator** scores each row's output. Our contract is:

```
evaluator(output: str, dataset_row: dict) -> EvaluationResult
```

`EvaluationResult` is the canonical type — it carries a `label`, a numeric `score`, and an `explanation`. Those land back on the experiment as `eval.<name>.label`, `eval.<name>.score`, and `eval.<name>.explanation` columns.

### Why a deterministic evaluator (not LLM-as-a-judge)?

We could use an [LLM-as-a-judge](https://arize.com/docs/ax/concepts/evaluators/online-llm-as-judge) here to score itineraries subjectively ("is this a good plan?"). For a tutorial, that's expensive, non-deterministic, and harder to reason about. A code evaluator is reproducible, free to run, and easy to read.

The trade-off: a deterministic eval can only check what you can describe in code. Subjective qualities like "is this engaging?" or "does it match the user's vibe?" need an LLM judge. In practice you'd often combine both.

### What we'll score

We want outputs that follow a strict format we can deploy: **`Day N: HH:MM - Activity - $cost` rows, every day, every cost**. The evaluator checks three signals, each worth a third of the score:

1. **Days covered** — does the output reference every day in the requested duration (Day 1, Day 2, ...)?
1. **Time stamps** — does it include `HH:MM` markers (at least one per day)?
1. **Cost figures** — does it include `$N` markers attached to activities (at least one per day)?

A vague v1 prompt will produce nice prose but miss the time and cost markers. A disciplined v2 prompt will hit all three. That's the comparison we're setting up.

In [ ]:
import re
from arize.experiments.evaluators.types import EvaluationResult


def itinerary_structure_eval(output, dataset_row) -> EvaluationResult:
    """Score: how closely the output follows the strict 'Day N: HH:MM - Activity - $cost' shape.

    Three signals, each worth a third:
      1. Every day mentioned (Day 1, Day 2, ...).
      2. At least one HH:MM time stamp per day.
      3. At least one $cost figure per day.

    Returns an EvaluationResult — the canonical evaluator return type. The SDK's experiment
    runner will turn this into eval.<name>.{label,score,explanation} columns on the result
    dataframe and on the experiment record in the AX UI.
    """
    try:
        # Defensive: a failing task can return None or a non-string. Score it 0 and move on
        # rather than crashing the whole experiment.
        if not output or not isinstance(output, str):
            return EvaluationResult(label="empty", score=0.0, explanation="Empty or non-string output.")

        # The experiment runner passes a ReadOnly view rather than a plain dict, so normalize
        # before doing dict-style access.
        row = dataset_row if isinstance(dataset_row, dict) else dict(dataset_row)

        # Parse the duration string ("3 days", "5 days") to the number of days we expect to see.
        duration = str(row.get("duration", ""))
        match = re.search(r"\d+", duration)
        n_days = int(match.group()) if match else 1

        # ---------- Signal 1: every day mentioned ----------
        # Look for "Day 1", "Day 2", ... up to n_days. \b ensures word boundaries so
        # "Day 12" doesn't accidentally satisfy n_days=1.
        days_present = sum(
            1 for d in range(1, n_days + 1)
            if re.search(rf"\bDay\s*{d}\b", output, re.IGNORECASE)
        )
        days_coverage = days_present / n_days  # 0.0 to 1.0

        # ---------- Signal 2: HH:MM time stamps ----------
        # Matches "9:00", "09:30", "14:45", etc. Requiring at least one per day on average
        # filters out outputs that only use "Morning/Afternoon" section headers. This is
        # the signal that reliably separates v1 (prose with vague time-of-day labels) from
        # v2 (which is forced into "Day N: HH:MM - ..." rows).
        time_markers = len(re.findall(r"\b\d{1,2}:\d{2}\b", output))
        has_times = time_markers >= n_days

        # ---------- Signal 3: dollar cost figures ----------
        # \$\d matches '$' followed immediately by a digit ($30, $80, $250, etc.).
        # This catches both per-activity costs ('Hagia Sophia - $30') AND prices in budget
        # summaries ('$80-120/night'). For this tutorial that's fine — verbose v1 outputs
        # tend to include budget tables that pass this check, so signal 3 alone isn't the
        # discriminator across our v1 vs v2 comparison; signal 2 above usually is.
        cost_markers = len(re.findall(r"\$\d", output))
        has_costs = cost_markers >= n_days

        # Combined score: average the three signals so each contributes equally. A perfect
        # output scores 1.0; missing either time stamps or cost figures caps the score at
        # 0.67 (since days_coverage of 1.0 + one zero + one one = 2.0/3).
        score = (days_coverage + (1.0 if has_times else 0.0) + (1.0 if has_costs else 0.0)) / 3

        if score >= 0.9:
            label = "well_structured"
        elif score >= 0.5:
            label = "partial"
        else:
            label = "poor"

        # The explanation is what shows up next to the score in the AX UI — write it so
        # someone reading a low score can immediately see which signal failed and why.
        return EvaluationResult(
            label=label,
            score=round(float(score), 2),
            explanation=(
                f"days:{days_present}/{n_days} | "
                f"times:{time_markers} ({'pass' if has_times else 'fail'}) | "
                f"costs:{cost_markers} ({'pass' if has_costs else 'fail'})"
            ),
        )
    except Exception as e:
        # Catch-all so a buggy evaluator doesn't sink the whole experiment. The
        # experiment record will show label=error with the exception text.
        return EvaluationResult(label="error", score=0.0, explanation=f"Eval crashed: {e}")

### Run the first experiment

`client.experiments.run(...)` is the workhorse. Given a dataset, a task, and a dict of evaluators, it:

1. Iterates the dataset, calling the task with each row.
1. Runs every evaluator on every output.
1. Writes an experiment record to AX (visible under the dataset's Experiments tab).
1. Returns the experiment metadata and a pandas DataFrame with one row per dataset row.

The returned DataFrame's columns include:

- `example_id` — the dataset row's ID.
- `output` — what the task returned.
- `error` — any task-level exception (None if the task succeeded).
- `result.trace.*` — the trace recorded for the task call.
- `eval.<eval-name>.{score,label,explanation,trace.*}` — one set of columns per evaluator you attached.

The cell below picks the score and label columns out of the DataFrame by suffix so the notebook keeps working even if Arize renames them in a future release.

In [ ]:
experiment_v1, results_v1 = client.experiments.run(
    name=f"trip-planner-v1-{RUN_ID}",  # unique experiment name; shows up in the UI under the dataset
    dataset=dataset.name,
    space=ARIZE_SPACE_ID,
    task=task_v1,
    evaluators={"itinerary_structure": itinerary_structure_eval},  # dict key becomes the eval column prefix
    concurrency=3,                                                  # how many task calls to run in parallel
    exit_on_error=True,                                             # surface task/evaluator errors loudly instead of silently scoring None
)

print(f"Columns: {list(results_v1.columns)}")
print()
print("Per-row scores (v1):")

# Find the eval columns by suffix — actual column name format may vary by SDK version,
# but they always end in .score / .label.
score_col = next((c for c in results_v1.columns if c.endswith("score") and "itinerary" in c), None)
label_col = next((c for c in results_v1.columns if c.endswith("label") and "itinerary" in c), None)
if score_col and label_col:
    print(results_v1[[score_col, label_col]].to_string())
    print(f"\nMean score: {results_v1[score_col].mean():.2f}")
else:
    # Defensive fallback if the column names shift — print enough to debug.
    print(results_v1.head().to_string())

## Section 3 — Iterate and compare

The v1 experiment surfaced a clear pattern: outputs look like helpful prose but don't match the strict shape our evaluator expects. The model added `Day N` headers (signal 1 passes) and budget summaries with `$` figures (signal 3 usually passes, since `$80-120` in a budget table satisfies the regex too) — but it consistently missed `HH:MM` time stamps on activities (signal 2 fails). That's the failure pattern we'll fix.

To fix it we'll **add a second version of the prompt with explicit format rules**, save it to Prompt Hub, tag it as `production`, and rerun the same experiment. Because v1 still exists as an immutable version, we can compare the two side by side without losing the old one.

### Two important moves in this section

1. **`client.prompts.create_version(...)`** — a new version of the *same* Prompt Object, not a new prompt. The original `prompt_v1.id` is the prompt identity; saving a version under that ID accrues to the same version history in Prompt Hub.
2. **`client.prompts.set_labels(...)`** — tags are mutable pointers at versions. Moving the `production` tag from v1 to v2 is how you "deploy" a new prompt without touching application code. See [Versioning and tags](https://arize.com/docs/ax/concepts/prompts/versioning-and-tags) for the conceptual model.

In [ ]:
# v2 system message: tightened to enforce the exact format our evaluator scores against.
# Key changes vs v1:
#   - Names the exact output shape: "Day N: HH:MM - Activity - $cost"
#   - Uses "MUST" / "ALWAYS" — concrete, unambiguous instructions outperform softer wording
#   - Tells the model what NOT to do ("no preamble or epilogue") to suppress the chatty intros v1 produced
system_message_v2 = (
    "You are a disciplined trip planner. For every day of the trip you MUST output:\n"
    "  Day N: HH:MM - Activity - $cost\n"
    "Use one bullet per time slot. ALWAYS include a $cost figure (estimate if exact "
    "price unknown). Cover the full duration. Be concise; no preamble or epilogue."
)

# create_version vs create: we pass prompt=prompt_v1.id so this becomes a new version of
# the SAME prompt object — same name, same history, just a new immutable snapshot.
prompt_v2 = client.prompts.create_version(
    prompt=prompt_v1.id,
    space=ARIZE_SPACE_ID,
    commit_message="v2: enforce strict day/time/cost format, drop preamble",
    input_variable_format=InputVariableFormat.F_STRING,
    provider=LlmProvider.OPEN_AI,
    model="gpt-5.4-mini",
    messages=[
        LLMMessage(role="system", content=system_message_v2),
        LLMMessage(role="user", content=user_message),  # user template is unchanged
    ],
    # v2 keeps the same invocation params as v1; the change under test is the system prompt.
    invocation_params=InvocationParams(max_completion_tokens=600),
)

print(f"Saved v2 (version id: {prompt_v2.id})")

In [ ]:
# Tag v2 as production. Tags are mutable pointers at versions — when an application calls
# client.prompts.get(prompt=PROMPT_NAME, label="production"), it gets whatever version
# this tag currently points at. Moving the tag is the deployment.
#
# v1 is still in Prompt Hub and recoverable — it just no longer carries the production
# label. A future v3 can take it next.
client.prompts.set_labels(version_id=prompt_v2.id, labels=["production"])
print("v2 tagged as production.")

In [ ]:
# Build the v2 task from the new system message. Everything else (user template, model,
# evaluator, dataset) is identical — that's exactly what makes the experiment comparable.
task_v2 = make_task(system_message_v2, user_message)

experiment_v2, results_v2 = client.experiments.run(
    name=f"trip-planner-v2-{RUN_ID}",
    dataset=dataset.name,       # SAME dataset as v1 — required for apples-to-apples comparison
    space=ARIZE_SPACE_ID,
    task=task_v2,
    evaluators={"itinerary_structure": itinerary_structure_eval},  # SAME evaluator as v1
    concurrency=3,
    exit_on_error=True,
)

print("Per-row scores (v2):")
score_col_v2 = next((c for c in results_v2.columns if c.endswith("score") and "itinerary" in c), None)
label_col_v2 = next((c for c in results_v2.columns if c.endswith("label") and "itinerary" in c), None)
if score_col_v2 and label_col_v2:
    print(results_v2[[score_col_v2, label_col_v2]].to_string())
    print(f"\nMean score: {results_v2[score_col_v2].mean():.2f}")
else:
    print(results_v2.head().to_string())

In [ ]:
# Build a side-by-side comparison frame. Because v1 and v2 ran on the SAME dataset rows
# in the SAME order, we can compare element-wise — row 0 of v1 corresponds to row 0 of v2.
score_col = next(c for c in results_v1.columns if c.endswith("score") and "itinerary" in c)
delta = pd.DataFrame({
    "v1_score": results_v1[score_col].values,
    "v2_score": results_v2[score_col].values,
})

# Positive delta = v2 improved on that row. Negative delta = v2 regressed on that row.
# In a real iteration cycle you want most rows positive AND no row strongly negative —
# regressing on a single edge case is how prompt edits ship silent bugs.
delta["delta"] = delta["v2_score"] - delta["v1_score"]
print(delta.to_string())
print(f"\nMean v1: {delta['v1_score'].mean():.2f}")
print(f"Mean v2: {delta['v2_score'].mean():.2f}")

### Reading the comparison

Three things to look at in the table above:

- **Mean v2 > mean v1.** The aggregate improved. If you re-ran this notebook with a stronger v2, you'd expect this to grow; if a future v3 introduces a regression on some row, you'd see the mean dip even if most rows still pass.
- **Every row's delta is non-negative.** Critically, v2 didn't regress on *any* row — no edge case got worse. This is what you want to confirm before promoting a version to production.
- **Some rows have larger deltas than others.** Look at the row with the largest positive delta — that's typically where v1 was struggling most. In a real workflow you'd open that row's trace in AX, read the v1 vs v2 outputs side by side, and confirm the improvement was the kind you wanted (not, say, the model truncating output to satisfy the regex).

### Open the comparison in the AX UI

This same comparison is available visually in Arize AX:

1. Navigate to your dataset (the one created above).
1. Open the **Experiments** tab — you'll see `trip-planner-v1-<RUN_ID>` and `trip-planner-v2-<RUN_ID>`.
1. Select both and click **Compare** — AX renders the per-row outputs, evaluator scores, and explanations side by side, with diff highlighting on the text outputs.

The UI is where you'd typically *read* the comparison. The notebook gave you the structure; the UI gives you the explanations and the per-row diffs that tell you *why* the scores moved.

## What you built

In ~20 cells you walked the full prompt iteration cycle:

1. Defined a **Prompt Object** with a template, model, and invocation parameters; saved it to Prompt Hub as immutable v1.
1. Built a small **dataset** of trip-planner inputs.
1. Defined a **task** (renders the prompt + calls the LLM) and a deterministic **evaluator** (scores the output's structural fitness).
1. Ran v1 as an experiment and saw it score well below 1.0 — outputs had `Day N` headers and budget tables, but lacked the `HH:MM` time stamps our evaluator expects on each activity.
1. Tightened the prompt into v2 with explicit format rules; saved it as a new version; tagged it `production`.
1. Re-ran the same dataset through v2 and confirmed every row improved (no regressions).

The same loop scales to real applications. The dataset gets bigger and more carefully chosen. The evaluator gets stricter or combines multiple signals (often including an LLM-as-a-judge for subjective qualities). The prompt iterations are driven by failure patterns surfaced in the experiment results.

## Next steps

- **Open both versions in Prompt Hub** to see them side by side, diff the templates, and inspect the version history. Your application can fetch by tag via `client.prompts.get(prompt=PROMPT_NAME, label="production")` — see [Loading prompts in your application](https://arize.com/docs/ax/concepts/prompts/loading-in-applications).
- **Open the experiments tab** to compare the two runs row by row, complete with per-row outputs, evaluator explanations, and the LLM traces behind each output.
- **Add an LLM-as-a-judge evaluator** for the subjective dimensions (helpfulness, tone, completeness) the deterministic eval can't catch. See [Evaluators](https://arize.com/docs/ax/concepts/evaluators/overview).
- **Wire the experiment into CI/CD** so a prompt edit becomes a PR check — block merges that regress the score. See [Prompts in CI/CD](https://arize.com/docs/ax/concepts/prompts/prompts-in-ci-cd).
- **Try Prompt Learning** for automated optimization once you have a golden dataset — the platform can search the prompt space for you. See [Optimizing prompts](https://arize.com/docs/ax/concepts/prompts/optimizing-prompts).

The pattern transfers directly to your own prompts: define what "good output" looks like as an evaluator, build a small golden dataset, then iterate — knowing that every score change shows up immediately in your experiment compare view.